<a href="https://colab.research.google.com/github/MarkChernov123/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']

# Calculate totals
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Units Sold: {total_units:,}")

Total Revenue: $8,520.00
Total Units Sold: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO

by_category = (df.groupby('category')['revenue'].sum().sort_values(ascending=False).to_frame())
by_category['share_pct'] = (by_category['revenue'] / df['revenue'].sum()) * 100

print(by_category)

          revenue  share_pct
category                    
Food       4293.0  50.387324
Merch      1771.5  20.792254
Drink      1554.0  18.239437
RainGear    901.5  10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
vendor_avg = (df.groupby('vendor_id')['revenue'].mean().sort_values(ascending=False).to_frame())

print(vendor_avg)

             revenue
vendor_id           
V-01       22.595745
V-18       21.750000
V-05       20.580645
V-10       20.314286


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_share = (
    df[df['category'] == 'Merch']['revenue'].sum() / df['revenue'].sum()
) * 100
print(f'{merch_share:.1f}%')

20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

#since vendor is unmatched, something doesnt line up so we need to use left join
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

joined.loc[joined['vendor_name'].isna(), 'vendor_name'] = 'Unlisted Vendor'

# 3. Check totals
print("Rows:", len(joined))
print("Total Revenue:", joined['revenue'].sum())

Rows: 400
Total Revenue: 8520.0


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
pivot = pd.pivot_table(joined,values='revenue',index='vendor_name',columns='category',aggfunc='sum',fill_value=0,margins=True,margins_name='Total',)

print(pivot)

category          Drink    Food   Merch  RainGear   Total
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
Unlisted Vendor   582.0  1018.5   508.5     240.0  2349.0
Total            1554.0  4293.0  1771.5     901.5  8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

_your answer here_

A) The vendors should prioritize increasing food inventory as they made the most revenue from food sales at a 50.4% share count than any other category of goods. Additionally, they could improve the amount of stations that high value vendors can sell at, such as unlisted vendor with had the most revenue of 2349. Improving the chances that the high performing vendors can sell more good would improve net revenue.

B) The most unreliable is Q5 due to the merging process I have set up only being able to be run once. It is not idempotent. The weakness comes from merging columns with a left join and it's difficult to merge tables that are already merged, since there's nothing to do, so the entire session has to be reset.